<a href="https://colab.research.google.com/github/ced-sys/SubTerra/blob/main/ee_colab(change_analysis%2C_time).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install earthengine-api folium

In [ ]:
import ee
import folium
import numpy as np

In [ ]:
ee.Authenticate()
ee.Initialize(project="subterra-466121")

In [ ]:
dataset=ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')

In [ ]:
point=ee.Geometry.Point(36.2695525, -0.8859869)

In [ ]:
image1=(dataset
        .filterDate('2023-01-01','2024-01-01')
        .filterBounds(point)
        .first())

In [ ]:
image2=(dataset
        .filterDate('2024-01-01', '2025-01-01')
        .filterBounds(point)
        .first())

In [ ]:
grid_size=1000

In [ ]:
bbox=point.buffer(grid_size/2).bounds()

image1_cropped=image1.clip(bbox)
image2_cropped=image2.clip(bbox)

print(f"Grid cell center:{point.getInfo()['coordinates']}")
print(f"Grid cell size: {grid_size}meters")

In [ ]:
dot_prod=(image1
          .multiply(image2_cropped)
          .reduce(ee.Reducer.sum()))

euclidean_distance=(image1_cropped
                    .subtract(image2_cropped)
                    .pow(2)
                    .reduce(ee.Reducer.sum())
                    .sqrt())

change_magnitude=(image1_cropped
                  .subtract(image2_cropped)
                  .abs()
                  .reduce(ee.Reducer.sum()))

mag1=image1_cropped.pow(2).reduce(ee.Reducer.sum()).sqrt()
mag2=image2_cropped.pow(2).reduce(ee.Reducer.sum()).sqrt()

cosine_similarity=dot_prod.divide(mag1.multiply(mag2))

change_threshold=0.3
significant_change=euclidean_distance.gt(change_threshold)

In [ ]:
map_center=[-0.8859869, 36.2695525]
m=folium.Map(location=map_center, zoom_start=12, tiles='OpenStreetMap')

In [ ]:
def add_ee_layer(ee_image_object, vis_params, name):
  map_id_dict=ee.Image(ee_image_object).getMapId(vis_params)
  folium.raster_layers.TileLayer(
      tiles=map_id_dict['tile_fetcher'].url_format,
      attr='Map Data &copy; <a href="https://earthengine.google.com/">Google Earth Engine</a>',
      name=name,
      overlay=True,
      control=True
  ).add_to(m)

In [ ]:
vis_params_embeddings={'min':-0.3, 'max':0.3, 'bands':['A01', 'A16', 'A09']}
vis_params_similarity={'min':0, 'max':1, 'palette':['red', 'yellow', 'green']}
vis_params_distance={'min':0, 'max':2, 'palette':['blue', 'white', 'red']}
vis_params_change_mag={'min':0, 'max':5, 'palette':['darkblue', 'blue', 'cyan', 'yellow', 'orange']}
vis_params_binary_change={'min':0, 'max':1, 'palette':['white', 'red']}


In [ ]:
add_ee_layer(image1_cropped, vis_params_embeddings, '2023 embeddings (grid cell)')
add_ee_layer(image2_cropped, vis_params_embeddings, '2024 embeddings (grid cell)')
add_ee_layer(dot_prod, vis_params_similarity, 'Similarity (green=more similar)')
add_ee_layer(euclidean_distance, vis_params_distance, 'Eucliadean Distance (red=more change)')
add_ee_layer(change_magnitude, vis_params_change_mag, 'Change Magnitude ')
add_ee_layer(significant_change, vis_params_binary_change, 'Significant Change Areas')


In [ ]:
def get_change_statistics():
  comprehensive_reducer=(ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True).combine(ee.Reducer.minMax(), sharedInputs=True).combine(ee.Reducer.count(), sharedInputs=True))

  similarity_stats=dot_prod.reduceRegion(comprehensive_reducer, bbox, 30, maxPixels=1e9)
  distance_stats=euclidean_distance.reduceRegion(comprehensive_reducer, bbox, 30, maxPixels=1e9)
  magnitude_stats=change_magnitude.reduceRegion(comprehensive_reducer, bbox, 30, maxPixels=1e9)
  cosine_stats=cosine_similarity.reduceRegion(comprehensive_reducer, bbox, 30, maxPixels=1e9)

  total_pixels=significant_change.select(['sum'], ['total']).reduceRegion(ee.Reducer.count(), bbox, 30, maxPixels=1e9)
  changed_pixels=significant_change.reduceRegion(ee.Reducer.sum(), bbox, 30, maxPixels=1e9)

  return {
      'similarity':similarity_stats.getInfo(),
      'euclidean_distance':distance_stats.getInfo(),
      'change_magnitude':magnitude_stats.getInfo(),
      'cosine_similarity':cosine_stats.getInfo(),
      'change_percentage':{
          'total_pixels':total_pixels.getInfo(),
          'changed_pixels':changed_pixels.getInfo()
      }
  }

In [ ]:
change_stats=get_change_statistics()

print("\n1. SIMILARITY METRICS:")
print("-Dot Product Similarity:", change_stats['similarity'])
print("-Cosine similarity:", change_stats['cosine_similarity'])

print("\n2. CHANGE/DISTANCE METRICS:")
print("-Euclidean Distance:", change_stats['euclidean_distance'])
print("-Change Magnitude:", change_stats['change_magnitude'])

print("\n3. CHANGE DETECTION:")
change_pixels=change_stats['change_percentage']['changed_pixels'].get('sum', 0)
total_pixels=change_stats['change_percentage']['total_pixels'].get('total', 1)
change_percentage=(change_pixels/total_pixels)*100 if total_pixels>0 else 0
print(f"Pixels with significant change: {change_pixels}")
print(f"-Total pixels: {total_pixels}")
print(f"-Percentage of area changed: {change_percentage:.2f}%")

print(f"\n4. INTERPRETATION:")
avg_similarity=change_stats['similarity'].get('sum_mean', 0)
avg_distance=change_stats['euclidean_distance'].get('sum_mean', 0)

if avg_similarity>0.8:
  print("-High similarity: Minimal changes detected")
elif avg_similarity>0.6:
  print("-Moderate SIMILARITY: Some changes detected")
else:
  print("-Low similarity: Significant changes detected")

print(f"Average similarity score: {avg_similarity:.3f}")
print(f"Average change distance: {avg_distance:.3f}")

In [ ]:
def export_change_analysis():
  change_stack=ee.Image.cat([
      dot_prod.rename('similarity'),
      euclidean_distance.rename('euclidean_distance'),
      change_magnitude.rename('change_magnitude'),
      cosine_similarity.rename('cosine_similarity'),
      significant_change.rename('significant_change')
  ])

  task=ee.batch.Export.image.toDrive(
      image=change_stack,
      description='change_analysis_kenya_2023_2024',
      folder='SubTerra',
      region=bbox,
      scale=30,
      maxPixels=1e9
  )
  task.start()
  print("Change analysis export task started")

  export_change_analysis()

In [ ]:
def add_grid_boundary(map_obj, geometry, name="Grid Cell Boundary"):
  coords=geometry.getInfo()['coordinates'][0]

  folium_coords=[[coord[1], coord[0]] for coord in coords]

  folium.vector_layers.PolyLine(
      locations=folium_coords,
      color='yellow',
      weight=3,
      opacity=0.8,
      popup=name
  ).add_to(map_obj)

add_grid_boundary(m, bbox, "Grid Cell Boundary")

In [ ]:
folium.LayerControl().add_to(m)

In [ ]:
folium.Marker(
    location=map_center,
    popup='Point of Interest',
    icon=folium.Icon(color='red')
).add_to(m)

In [ ]:
m